#**Benchmark & Pipeline ML — NYC Taxi**
####Trabalho realizado para disciplina de Ciência de Dados em Larga Escala
#####**Grupo:** Renata Azevedo (up202512262), Giuliano (up202512089) e Ni (up202205842)

16.   **Experimento 5 - Profiling com cProfile**



O profiling detalhado foi realizado apenas sobre a implementação em Pandas.

Bibliotecas como Dask, Modin e RAPIDS executam grande parte das operações através de mecanismos internos de paralelização, processos externos ou kernels compilados em C/C++ e CUDA. Nestes casos, o cProfile não consegue capturar adequadamente o custo real das operações, produzindo resultados menos representativos e difíceis de comparar.

Desta forma, o profiling foi concentrado na implementação em Pandas, utilizada como baseline do projeto. A comparação entre bibliotecas foi realizada através dos tempos de execução medidos experimentalmente em cada tarefa, enquanto o cProfile foi utilizado para compreender em maior detalhe o comportamento interno da solução de referência.

>16.1.   **Configuração do Ambiente**


Instalação da biblioteca de processamento de dados:

In [22]:
# ============================================================
# INSTALAÇÕES — execute apenas uma vez no Colab
# Descomente as linhas necessárias e volte a comentar após instalar
# ============================================================


# !pip install yappi                           # Profiling multi-thread

print("OK.")

OK.


Iniciando a leitura com algumas importações necessárias:

In [23]:
#============================================================
# IMPORTS GERAIS
# ============================================================
import os, time, warnings, urllib.request, gc
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import cProfile
import pstats
import io

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

print(f"Pandas  : {pd.__version__}")
print(f"NumPy   : {np.__version__}")
print(f"Data    : {datetime.now().strftime('%Y-%m-%d %H:%M')}")

Pandas  : 2.2.2
NumPy   : 2.0.2
Data    : 2026-06-03 18:55


In [24]:
# ============================================================
# IMPORTS DAS BIBLIOTECAS BIG DATA
# try/except: o notebook não quebra se uma lib não estiver instalada
# ============================================================

# Injeta dicionário vazio para compatibilidade do Spark com Pandas recente
if not hasattr(pd.core.common, '_builtin_table'):
    pd.core.common._builtin_table = {}

# --- Dask ---
try:
    import dask.dataframe as dd
    import dask
    print(f"Dask        : {dask.__version__}")
    DASK_AVAILABLE = True
except ImportError:
    print("Dask não disponível")
    DASK_AVAILABLE = False

# --- Modin (Ray como backend default) ---
# [ADICIONADO] — Modin não estava no notebook original
#try:
#    import modin.pandas as mpd
#    import modin
 #   print(f" Modin       : {modin.__version__}")
#    MODIN_AVAILABLE = True
#except ImportError:
#    print(" Modin não disponível (pip install modin[ray])")
#    MODIN_AVAILABLE = False
#import ray


#if ray.is_initialized():
    #ray.shutdown()

# Força o Ray a respeitar os limites apertados do Colab gratuito
#  ray.init(
#    num_cpus=2,                     # O Colab gratuito só tem 2 vCPUs
#   object_store_memory=2 * 1024 * 1024 * 1024, # Limita o cache do Ray a 2 GB (evita estouro)
#   _memory=4 * 1024 * 1024 * 1024  # Limita a memória de trabalho a 4 GB
#)

# --- PySpark / Koalas ---
try:
    import pyspark.pandas as ps
    from pyspark.sql import SparkSession
    import pyspark
    print(f" PySpark     : {pyspark.__version__}")
    SPARK_AVAILABLE = True
except Exception as e:
    print(f" PySpark não disponível: {str(e)[:60]}")
    SPARK_AVAILABLE = False

# --- JobLib (já incluído no Scikit-learn, mas importamos explicitamente) ---
# [ADICIONADO] — JobLib não estava no notebook original
try:
    import joblib
    print(f" JobLib      : {joblib.__version__}")
    JOBLIB_AVAILABLE = True
except ImportError:
    print(" JobLib não disponível")
    JOBLIB_AVAILABLE = False

# --- RAPIDS cuDF (GPU) ---
# [ADICIONADO] — Rapids não estava no notebook original
try:
    import cudf
    print(f" cuDF (GPU)  : {cudf.__version__}")
    CUDF_AVAILABLE = True
except ImportError:
    print(" cuDF não disponível (sem GPU NVIDIA/CUDA)")
    CUDF_AVAILABLE = False

# --- XGBoost ---
# [ADICIONADO] — necessário para o Pipeline ML
try:
    import xgboost as xgb
    print(f" XGBoost     : {xgb.__version__}")
    XGB_AVAILABLE = True
except ImportError:
    print(" XGBoost não disponível (pip install xgboost)")
    XGB_AVAILABLE = False

# --- Scikit-learn ---
try:
    import sklearn
    print(f" Scikit-learn: {sklearn.__version__}")
except ImportError:
    print(" Scikit-learn não disponível")

Dask        : 2026.3.0
 PySpark     : 4.0.2
 JobLib      : 1.5.3
 cuDF não disponível (sem GPU NVIDIA/CUDA)
 XGBoost     : 3.2.0
 Scikit-learn: 1.6.1


In [25]:
# ============================================================
# CONFIGURAÇÃO GLOBAL
# Altere apenas estas variáveis para escalar o experimento
# ============================================================

ANO       = 2026   # Ano dos dados NYC Taxi
NUM_MESES = 12      # Número de meses a descarregar (1 = ~61 MB; 12 = ~730 MB)
caminho   = Path("./FileStore/taxi/csv2026")  # Pasta local dos dados

caminho.mkdir(parents=True, exist_ok=True)
print(f" Pasta: {caminho.resolve()}")
print(f" Período: {ANO}, meses 1 a {NUM_MESES}")

 Pasta: /content/FileStore/taxi/csv2026
 Período: 2026, meses 1 a 12


In [26]:
# Lista os ficheiros se a pasta existir
if caminho.exists():
    ficheiros = list(caminho.iterdir())
    if ficheiros:
        for f in ficheiros:
            print(f.name)
    else:
        print("A pasta está vazia.")
else:
    print("A pasta ainda não foi criada.")

yellow_tripdata_2026-02.parquet
yellow_tripdata_2026-01.parquet


In [27]:
opener = urllib.request.build_opener()
opener.addheaders = [
    ('User-Agent',
     'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')
]
urllib.request.install_opener(opener)

In [28]:
# ============================================================
# DOWNLOAD AUTOMÁTICO DOS FICHEIROS PARQUET
# Parquet é um formato colunar — muito mais eficiente que CSV
# para as leituras parciais que o Dask/Spark fazem.
# ============================================================

url_loc = {} # Mapeia a URL de download para o caminho do ficheiro



for year in range(2026, 2027):
    for m in range(1, 3):
        month = "{:02d}".format(m)

        # 1. URL dinâmica com base nas variáveis do loop (Aponta para o .parquet real de cada mês)
        url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year}-{month}.parquet"

        # 2. Nome do ficheiro local atualizado para refletir a extensão correta (.parquet)
        fname = f"yellow_tripdata_{year}-{month}.parquet"

        # Como está no Windows (caminho atual), o destino aponta para a pasta local criada
        loc = f"./FileStore/taxi/csv2026/{fname}"

        # Guarda no dicionário
        url_loc[url] = loc

In [29]:
# 3. Executa o loop de download corrigido
for url, loc in url_loc.items():
    if Path(loc).exists():
        mb = Path(loc).stat().st_size / 1e6
        print(f" Já existe: {Path(loc).name} ({mb:.1f} MB)")
        continue
    try:
        print(f"⬇  A descarregar: {url} ...")
        urllib.request.urlretrieve(url, loc)
        mb = Path(loc).stat().st_size / 1e6
        print(f"   Guardado: {mb:.1f} MB")
    except Exception as e:
        print(f" Erro: {e}")

print("Processo de download concluído!")


 Já existe: yellow_tripdata_2026-01.parquet (64.2 MB)
 Já existe: yellow_tripdata_2026-02.parquet (58.7 MB)
Processo de download concluído!


16.2   **Início das Operações**

Replicamos o benchmark do blog Databricks (2021) nas 5 operações principais.


In [30]:
resultados = []   # lista global: (operação, biblioteca, tempo_s)

In [31]:
def benchmark(name, func):
    start = time.time()
    result = func()
    end = time.time()

    print(f"{name}: {end - start:.4f} sec")
    return result

In [32]:
def benchmark_pandas(folder_path: Path):
    print("Iniciando leitura com Pandas...")

    start = time.perf_counter()

    dfs = []
    for f in folder_path.glob("*.parquet"):
        dfs.append(pd.read_parquet(f))

    df = pd.concat(dfs, ignore_index=True)

    elapsed = time.perf_counter() - start

    print("\nResultado Pandas:")
    print(f"- Linhas: {len(df)}")
    print(f"- Colunas: {df.shape[1]}")
    print(f"- Tempo: {elapsed:.2f}s")
    resultados.append(('Task 1 - Leitura', 'Pandas', round(elapsed, 4)))  # substitui com o valor real
    return df, elapsed

In [33]:
def benchmark_pandas_cprofile(caminho):

    pr = cProfile.Profile()
    pr.enable()

    start = time.perf_counter()

    df = pd.read_parquet(caminho)

    t = time.perf_counter() - start

    pr.disable()

    # ============================================================
    # MOSTRAR PROFILE
    # ============================================================

    s = io.StringIO()
    stats = pstats.Stats(pr, stream=s).sort_stats('cumtime')
    stats.print_stats(10)

    print(s.getvalue())

    return df, t

In [34]:
# Define o caminho da pasta local onde os ficheiros estão a ser guardados
caminho = Path("./FileStore/taxi/csv2026")

# Lista o conteúdo da pasta
if caminho.exists():
    ficheiros = list(caminho.iterdir())
    if ficheiros:
        print(f"Encontrados {len(ficheiros)} ficheiro(s):")
        for f in ficheiros:
            print(f"- {f.name}")
    else:
        print("A pasta está vazia. Precisa de executar o loop de download primeiro!")
else:
    print("A pasta ainda não existe no caminho atual.")

Encontrados 2 ficheiro(s):
- yellow_tripdata_2026-02.parquet
- yellow_tripdata_2026-01.parquet



16.2.1  **Task 1 — Leitura dos Dados (*Reading*)**


In [35]:
df, t_pandas = benchmark_pandas_cprofile(caminho)
resultados.append(('Task 1 - Leitura', 'Cprofile', round(t_pandas, 4)))  # substitui com o valor real
df.head()

         1413 function calls (1381 primitive calls) in 3.025 seconds

   Ordered by: cumulative time
   List reduced from 285 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      2/1    0.001    0.000    2.828    2.828 /usr/local/lib/python3.12/dist-packages/pandas/io/parquet.py:498(read_parquet)
      2/1    0.001    0.000    2.827    2.827 /usr/local/lib/python3.12/dist-packages/pandas/io/parquet.py:239(read)
        2    1.949    0.974    1.950    0.975 {built-in method time.sleep}
      2/1    0.735    0.367    0.790    0.790 /usr/local/lib/python3.12/dist-packages/pyarrow/parquet/core.py:1776(read_table)
       10    0.000    0.000    0.198    0.020 /usr/lib/python3.12/asyncio/base_events.py:1922(_run_once)
        1    0.085    0.085    0.086    0.086 /usr/local/lib/python3.12/dist-packages/pyarrow/pandas_compat.py:769(table_to_dataframe)
        1    0.055    0.055    0.055    0.055 /usr/local/lib/python3.12/dist-packages/p

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75


16.2.1.1  **Análise de Profiling — Leitura de Dados**

O profiling da operação de leitura foi realizado utilizando a ferramenta cProfile, permitindo identificar as funções responsáveis pela maior parte do tempo de execução.

O resultado mostrou que a leitura do ficheiro Parquet consumiu aproximadamente 2.063 segundos, sendo praticamente todo o tempo concentrado em funções da biblioteca PyArrow, utilizada internamente pelo Pandas para processar ficheiros Parquet.

| Função               | Tempo acumulado (s) |
| -------------------- | ------------------- |
| read_parquet         | 2.063               |
| pyarrow.read_table   | 1.250               |
| pyarrow.parquet.read | 0.548               |
| table_to_dataframe   | 0.359               |
| time.sleep           | 0.452               |
| select (epoll)       | 0.158               |


Observa-se que:

A função pyarrow.parquet.read_table() consumiu aproximadamente 1.25 segundos, correspondendo à leitura efetiva dos dados armazenados no ficheiro Parquet.


A função pyarrow.pandas_compat.table_to_dataframe() consumiu aproximadamente 0,359 segundos, correspondendo à conversão da estrutura interna do PyArrow para um DataFrame Pandas.

>16.2.2  **Task 2 — Agregações e Filtragem (*Aggregation & Filtering*)**


**Primeiro experimento:** Contagem de Valores. Contar quantas viagens ocorreram por tipo de pagamento.

In [36]:
print("─" * 45)
print("Task 2A — VALUE COUNTS (Pandas + cProfile)")
print("─" * 45)

# ============================================================
# PROFILING SETUP
# ============================================================

profiler = cProfile.Profile()
profiler.enable()

# ============================================================
# OPERAÇÃO + BENCHMARK
# ============================================================

start_p2 = time.perf_counter()

counts_pandas = df['VendorID'].value_counts()

t_p_vc = time.perf_counter() - start_p2

profiler.disable()

# ============================================================
# OUTPUT DO PROFILE
# ============================================================

stream = io.StringIO()
stats = pstats.Stats(profiler, stream=stream).sort_stats('cumtime')
stats.print_stats(10)

print(stream.getvalue())

# ============================================================
# RESULTADO
# ============================================================

print(f"Tempo Value Counts (Pandas) : {t_p_vc:.4f}s")
print(counts_pandas)

resultados.append((
    'Task 2a - Value Counts',
    'Pandas + cProfile',
    round(t_p_vc, 4)
))

─────────────────────────────────────────────
Task 2A — VALUE COUNTS (Pandas + cProfile)
─────────────────────────────────────────────
         652 function calls (643 primitive calls) in 0.087 seconds

   Ordered by: cumulative time
   List reduced from 222 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      5/4    0.000    0.000    0.002    0.000 /usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3512(run_code)
      5/4    0.084    0.017    0.002    0.000 {built-in method builtins.exec}
        1    0.000    0.000    0.001    0.001 /tmp/ipykernel_2718/3675823072.py:1(<cell line: 0>)
        1    0.000    0.000    0.001    0.001 /usr/local/lib/python3.12/dist-packages/pandas/core/base.py:923(value_counts)
        1    0.000    0.000    0.001    0.001 /usr/local/lib/python3.12/dist-packages/pandas/core/algorithms.py:862(value_counts_internal)
        1    0.000    0.000    0.001    0.001 /usr/local/lib/pyth

>**Análise de Profiling — Value Counts (Pandas)**

O profiling da operação Value Counts em Pandas revelou um tempo total de execução de 0.093 segundos, indicando uma operação extremamente eficiente mesmo em datasets de grande dimensão.

| Função                  | Tempo acumulado (s) |
| ----------------------- | ------------------- |
| value_counts()          | 0.092               |
| value_counts_internal() | 0.092               |
| sort_values()           | 0.006               |
| nargsort()              | 0.005               |
| numpy.ndarray.argsort() | 0.005               |



A análise mostra que:

O núcleo da operação (value_counts_internal) é extremamente rápido (~0.006s).



O principal custo computacional está na ordenação dos resultados, realizada via numpy.argsort(), que representa a parte mais pesada do pipeline.



O overhead de Python é praticamente irrelevante, sendo quase toda a execução delegada a operações vetorizadas em NumPy.


**Segundo experimento:** GroupBy. Vamos calcular o valor médio da tarifa para cada tipo de fornecedor.

In [37]:
print("─" * 45)
print("Task 2B — GROUPBY MEAN (Pandas + cProfile)")
print("─" * 45)

# ============================================================
# PROFILING SETUP
# ============================================================

profiler = cProfile.Profile()
profiler.enable()

# ============================================================
# OPERAÇÃO + BENCHMARK
# ============================================================

start_p3 = time.perf_counter()

mean_pandas = df.groupby('VendorID')['fare_amount'].mean()

t_p_gb = time.perf_counter() - start_p3

profiler.disable()

# ============================================================
# OUTPUT DO PROFILE
# ============================================================

stream = io.StringIO()
stats = pstats.Stats(profiler, stream=stream).sort_stats('cumtime')
stats.print_stats(10)

print(stream.getvalue())

# ============================================================
# RESULTADO
# ============================================================

print(f"Tempo GroupBy (Pandas) : {t_p_gb:.4f}s")
print(mean_pandas)

resultados.append((
    'Task 2b - GroupBy',
    'Pandas + cProfile',
    round(t_p_gb, 4)
))

─────────────────────────────────────────────
Task 2B — GROUPBY MEAN (Pandas + cProfile)
─────────────────────────────────────────────
         1504 function calls (1486 primitive calls) in 0.259 seconds

   Ordered by: cumulative time
   List reduced from 333 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.010    0.010    0.156    0.156 /usr/local/lib/python3.12/dist-packages/pandas/core/groupby/ops.py:735(has_dropped_na)
        1    0.000    0.000    0.145    0.145 /usr/local/lib/python3.12/dist-packages/pandas/core/groupby/ops.py:743(group_info)
        1    0.000    0.000    0.145    0.145 /usr/local/lib/python3.12/dist-packages/pandas/core/groupby/ops.py:758(_get_compressed_codes)
        1    0.003    0.003    0.144    0.144 /usr/local/lib/python3.12/dist-packages/pandas/core/groupby/grouper.py:689(codes)
        1    0.000    0.000    0.125    0.125 /usr/local/lib/python3.12/dist-packages/pandas/core/groupby/

>**Análise de Profiling — GroupBy Mean (Pandas)**

O profiling da operação GroupBy + Mean revelou um tempo total de execução de 0.2831 segundos, sendo a operação mais pesada analisada até ao momento neste conjunto de tarefas.

| Função                | Tempo acumulado (s) |
| --------------------- | ------------------- |
| groupby.mean()        | 0.282               |
| _cython_agg_general   | 0.282               |
| grouped_reduce        | 0.282               |
| array_func            | 0.281               |
| _cython_operation     | 0.281               |
| group_info            | 0.16               |
| has_dropped_na()| 0.175s


O GroupBy em Pandas é uma operação mais complexa do que parece:

-  Construção dos grupos
group_info() + codes()
cria mapeamento entre linhas e grupos
envolve hashing + compressão de categorias

 Isto explica ~50% do custo total.

 **Terceiro experimento:** Filtragem Complexa. Vamos filtrar apenas as viagens que tiveram mais de 2 passageiros e calcular a distância média percorrida para esse grupo específico.

In [38]:
print("─" * 45)
print("Task 2C — FILTRAGEM (Pandas + cProfile)")
print("─" * 45)


# ============================================================
# PROFILING SETUP
# ============================================================

profiler = cProfile.Profile()
profiler.enable()

# ============================================================
# OPERAÇÃO + BENCHMARK
# ============================================================

start_p4 = time.perf_counter()

filtro_p = df[df['passenger_count'] > 2]['trip_distance'].mean()

t_p_fi = time.perf_counter() - start_p4

profiler.disable()

# ============================================================
# OUTPUT DO PROFILE
# ============================================================

stream = io.StringIO()
stats = pstats.Stats(profiler, stream=stream).sort_stats('cumtime')
stats.print_stats(10)

print(stream.getvalue())

# ============================================================
# RESULTADO
# ============================================================

print(f"Tempo Filtragem (Pandas) : {t_p_fi:.4f}s")
print(f"Média distância: {filtro_p:.4f}")

resultados.append((
    'Task 2c - Filtragem',
    'Pandas + cProfile',
    round(t_p_fi, 4)
))

─────────────────────────────────────────────
Task 2C — FILTRAGEM (Pandas + cProfile)
─────────────────────────────────────────────
         1349 function calls (1313 primitive calls) in 0.101 seconds

   Ordered by: cumulative time
   List reduced from 315 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        3    0.000    0.000    0.085    0.028 /usr/local/lib/python3.12/dist-packages/pandas/core/frame.py:4062(__getitem__)
        1    0.000    0.000    0.085    0.085 /usr/local/lib/python3.12/dist-packages/pandas/core/frame.py:4130(_getitem_bool_array)
        1    0.000    0.000    0.073    0.073 /usr/local/lib/python3.12/dist-packages/pandas/core/generic.py:4142(_take_with_is_copy)
        1    0.000    0.000    0.073    0.073 /usr/local/lib/python3.12/dist-packages/pandas/core/generic.py:4027(take)
        1    0.000    0.000    0.073    0.073 /usr/local/lib/python3.12/dist-packages/pandas/core/internals/managers.py:869(tak

**Análise de Profiling — Filtragem Complexa (Pandas)**


A análise do profiling da operação de filtragem mostra que o tempo total de execução foi de 0.145 segundos, sendo significativamente superior às operações de value_counts, mas inferior ao groupby.

| Função              | Tempo acumulado (s) |
| ------------------- | ------------------- |
| **getitem**         | 0.131               |
| _getitem_bool_array | 0.130               |
| _take_with_is_copy  | 0.118               |
| take                | 0.118               |
| reindex_indexer     | 0.114               |
| take_nd             | 0.114               |


O custo não está na operação de média final, mas sim na etapa de filtragem e seleção de linhas do DataFrame.

A maior parte do tempo é consumida por operações de manipulação de memória:

- _take_with_is_copy → 0.118s
- take → 0.118s
- reindex_indexer → 0.114s
- take_nd → 0.114s

 Estas funções indicam que o Pandas está a:

- criar um novo DataFrame
- copiar subconjuntos de dados da memória original
- reorganizar índices internamente

 ---
>16.2.3  **Task 3 — Join de Tabelas**


O benchmark Databricks faz um join para traduzir IDs em nomes de localização.

In [39]:
# ============================================================
# PREPARAÇÃO GLOBAL (executar apenas UMA vez)
# ============================================================

url_zonas = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

try:
    df_zones_p = pd.read_csv(url_zonas)
    df_zones_p = df_zones_p.rename(
        columns={'LocationID': 'PULocationID'}
    )

except Exception as e:
    print(f"Erro ao descarregar zonas online: {e}")

    df_zones_p = pd.DataFrame({
        'PULocationID': range(1, 266),
        'Zone': [f'Zone {i}' for i in range(1, 266)]
    })

# Parquets
caminhos = [str(f) for f in sorted(caminho.glob("*.parquet"))]

df_jan = pd.read_parquet(caminhos[0])
df_fev = pd.read_parquet(caminhos[1]) if len(caminhos) > 1 else df_jan

# Dataset combinado
df_viagens_p = pd.concat([df_jan, df_fev], join='outer')

In [40]:
# ============================================================
# PANDAS + cProfile
# ============================================================

print("─" * 45)
print("Task 3 — JOIN (Pandas + cProfile)")
print("─" * 45)

profiler = cProfile.Profile()
profiler.enable()

# ============================================================
# OPERAÇÃO
# ============================================================

start_p = time.perf_counter()

df_final_p = df_viagens_p.merge(
    df_zones_p,
    on='PULocationID',
    how='left'
)

_ = df_final_p.head(1)

t_p_join = time.perf_counter() - start_p

profiler.disable()

# ============================================================
# PROFILE OUTPUT
# ============================================================

stream = io.StringIO()
stats = pstats.Stats(profiler, stream=stream).sort_stats('cumtime')
stats.print_stats(10)

print(stream.getvalue())

# ============================================================
# RESULTADO
# ============================================================

print(f"Tempo Join (Pandas) : {t_p_join:.4f}s")

display(
    df_final_p[['PULocationID', 'Zone', 'fare_amount']].head(3)
)

resultados.append((
    'Task 3 - Join',
    'Pandas + cProfile',
    round(t_p_join, 4)
))

─────────────────────────────────────────────
Task 3 — JOIN (Pandas + cProfile)
─────────────────────────────────────────────
         3489 function calls (3401 primitive calls) in 3.130 seconds

   Ordered by: cumulative time
   List reduced from 536 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.052    0.052    2.629    2.629 /usr/local/lib/python3.12/dist-packages/pandas/core/reshape/merge.py:882(get_result)
        1    0.000    0.000    2.576    2.576 /usr/local/lib/python3.12/dist-packages/pandas/core/reshape/merge.py:825(_reindex_and_concat)
        1    0.000    0.000    2.308    2.308 /usr/local/lib/python3.12/dist-packages/pandas/core/reshape/concat.py:157(concat)
        1    0.000    0.000    2.308    2.308 /usr/local/lib/python3.12/dist-packages/pandas/core/reshape/concat.py:622(get_result)
        1    0.000    0.000    2.308    2.308 /usr/local/lib/python3.12/dist-packages/pandas/core/internals/concat

,PULocationID,Zone,fare_amount
0,239,Upper West Side South,7.2
1,163,Midtown North,7.9
2,43,Central Park,10.7


>>16.2.3.1**Análise de Resultado Join**

A operação de JOIN (merge) apresenta um tempo total de execução de 2.196 segundos, sendo  a operação mais pesada do pipeline analisado.

O profiling mostra claramente que o custo não está no merge em si, mas sim em várias etapas internas de alinhamento, concatenação e cópia de memória.





| Função                          | Tempo acumulado (s) |
| ------------------------------- | ------------------- |
| merge                           | 2.189               |
| get_result                      | 2.189               |
| _reindex_and_concat             | 1.618               |
| concat                          | 1.356               |
| concatenate_managers            | 1.356               |
| managers.copy                   | 1.355               |
| _maybe_reindex_columns_na_proxy | 1.355               |
| _consolidate_inplace            | 0.714               |
| _consolidate                    | 0.712               |


1. Etapa principal: merge + alinhamento

A função:

merge → 2.189s

depende fortemente de:

- alinhamento das chaves (PULocationID)
- criação de índices temporários
reorganização de linhas

2. Reindexação e concatenação (gargalo principal)

- _reindex_and_concat → 1.618s
- concat → 1.356s
- concatenate_managers → 1.356s

 Isto indica que o Pandas está a:

- alinhar os dois DataFrames
- reconstruir estruturas internas
- concatenar blocos de memória

3. Cópias de memória (bottleneck crítico)

 custo elevado aparece em:

- managers.copy → 1.355s
- _maybe_reindex_columns_na_proxy → 1.355s

Isto mostra que o Pandas está a:

- copiar blocos inteiros de dados
- evitar operações in-place
- realocar memória várias vezes

4. Consolidação de memória interna
- _consolidate_inplace → 0.714s
- _consolidate → 0.712s

Aqui o Pandas tenta:

- otimizar blocos de memória
- reduzir fragmentação interna

mas isso também tem custo elevado.


---
>16.2.4  **Task 4 — Cálculo Aritmético (*Arithmetic Calculation*)**

Cria nova coluna: gorjeta por passageiro (`tip_per_passenger = tip_amount / passenger_count`).

In [41]:
print("─" * 45)
print("Task 4 — CÁLCULO ARITMÉTICO (Pandas + cProfile)")
print("─" * 45)

# ============================================================
# PROFILING SETUP
# ============================================================

profiler = cProfile.Profile()
profiler.enable()

# ============================================================
# OPERAÇÃO
# ============================================================

start = time.perf_counter()

df['tip_per_passenger'] = (
    df['tip_amount'] /
    df['passenger_count'].replace(0, np.nan)
)

t_p_calc = time.perf_counter() - start

profiler.disable()

# ============================================================
# PROFILE OUTPUT
# ============================================================

stream = io.StringIO()
stats = pstats.Stats(profiler, stream=stream).sort_stats('cumtime')
stats.print_stats(10)

print(stream.getvalue())

# ============================================================
# RESULTADO
# ============================================================

print(f"Tempo Cálculo (Pandas) : {t_p_calc:.4f}s")

resultados.append((
    'Task 4 - Cálculo',
    'Pandas + cProfile',
    round(t_p_calc, 4)
))

─────────────────────────────────────────────
Task 4 — CÁLCULO ARITMÉTICO (Pandas + cProfile)
─────────────────────────────────────────────
         1897 function calls (1872 primitive calls) in 0.093 seconds

   Ordered by: cumulative time
   List reduced from 413 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.001    0.001    0.092    0.092 /tmp/ipykernel_2718/1293515259.py:1(<cell line: 0>)
        1    0.000    0.000    0.040    0.040 /usr/local/lib/python3.12/dist-packages/pandas/core/generic.py:7885(replace)
        1    0.001    0.001    0.040    0.040 /usr/local/lib/python3.12/dist-packages/pandas/core/internals/base.py:243(replace)
        2    0.039    0.019    0.039    0.019 {method 'copy' of 'numpy.ndarray' objects}
        1    0.000    0.000    0.037    0.037 /usr/local/lib/python3.12/dist-packages/pandas/core/internals/managers.py:317(apply)
        1    0.000    0.000    0.036    0.036 /usr/local/lib/

>>16.2.4.1 **Análise de Resultado Cálculo Aritmético (Pandas)**

O profiling da operação de cálculo aritmético revelou um tempo total de execução de aproximadamente 0,095 segundos, indicando que esta é uma operação relativamente eficiente no Pandas, embora com algum overhead associado à gestão interna de memória.

| Função                       | Tempo acumulado (s) |
| ---------------------------- | ------------------- |
| `replace()` (interno Pandas) | 0.040               |
| `numpy.ndarray.copy()`       | 0.036               |
| `_set_item()`                | 0.027               |
| `_arith_method()`            | 0.025               |
| `__truediv__()`              | 0.025               |
| `array_ops.arithmetic_op()`  | 0.025               |


A análise mostra que a maior parte do tempo não é consumida pelo cálculo matemático em si (divisão vetorizada), mas sim por operações internas de gestão de memória e estrutura do DataFrame.

A função:

- numpy.ndarray.copy()

foi uma das principais responsáveis pelo tempo de execução (~0,036s), indicando que o Pandas teve de criar cópias intermédias dos dados para garantir segurança e consistência das operações.

Além disso, funções como:

- _set_item()
- replace()

mostram que existe um custo relevante associado à atualização estrutural do DataFrame, incluindo alinhamento de índices e manipulação de blocos internos de memória.

---
>16.2.5  **Task 5 — Escrita dos Dados (*Writing / Persistence*)**

Persistência dos resultados em formato Parquet.

In [42]:
print("─" * 45)
print("Task 5 — ESCRITA EM PARQUET (Pandas + cProfile)")
print("─" * 45)

# ============================================================
# PROFILING SETUP
# ============================================================

profiler = cProfile.Profile()
profiler.enable()

# ============================================================
# OPERAÇÃO
# ============================================================

start = time.perf_counter()

df.to_parquet('./resultado_pandas.parquet', index=False)

t_p_write = time.perf_counter() - start

profiler.disable()

# ============================================================
# PROFILE OUTPUT
# ============================================================

stream = io.StringIO()
stats = pstats.Stats(profiler, stream=stream).sort_stats('cumtime')
stats.print_stats(10)

print(stream.getvalue())

# ============================================================
# RESULTADO
# ============================================================

print(f"Tempo Escrita (Pandas) : {t_p_write:.4f}s")

resultados.append((
    'Task 5 - Escrita',
    'Pandas + cProfile',
    round(t_p_write, 4)
))

─────────────────────────────────────────────
Task 5 — ESCRITA EM PARQUET (Pandas + cProfile)
─────────────────────────────────────────────
         6513 function calls (6440 primitive calls) in 4.836 seconds

   Ordered by: cumulative time
   List reduced from 429 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      5/4    0.000    0.000    4.835    1.209 /usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3512(run_code)
        4    0.000    0.000    4.835    1.209 {built-in method builtins.exec}
        1    0.000    0.000    4.835    4.835 /usr/local/lib/python3.12/dist-packages/pandas/util/_decorators.py:325(wrapper)
        1    0.000    0.000    4.835    4.835 /usr/local/lib/python3.12/dist-packages/pandas/core/frame.py:3018(to_parquet)
        1    0.000    0.000    4.834    4.834 /usr/local/lib/python3.12/dist-packages/pandas/io/parquet.py:409(to_parquet)
        1    0.000    0.000    4.830    4.830 

>>16.2.4.1 **Análise de Resultado Escrita em Parquet (Pandas)**

O profiling da operação de escrita em Parquet revelou um tempo total de execução de aproximadamente 5,00 segundos, sendo claramente a operação mais pesada do pipeline de processamento de dados.

| Função                               | Tempo acumulado (s) |
| ------------------------------------ | ------------------- |
| `to_parquet()`                       | 5.000               |
| `parquet.write()` (Pandas I/O layer) | 5.000               |
| `pyarrow.parquet.write_table()`      | 4.341               |
| `time.sleep()`                       | 4.001               |
| `threading.join()`                   | 1.286               |
| `threading.lock.acquire()`           | 1.288               |

Análise dos resultados

A análise mostra que a maior parte do tempo não está associada ao cálculo ou transformação de dados, mas sim ao processo de serialização e escrita em disco no formato Parquet.

A função:

pyarrow.parquet.write_table()

foi a principal responsável pelo tempo de execução (~4,34s)

Além disso, observa-se um tempo elevado em:

time.sleep()
threading.join()

o que indica forte presença de operações de sincronização e I/O blocking, típicas de escrita em disco ou pipeline de buffers internos.

**Próximo Notebook:**

 6-experimento_ML.ipynb - Pipeline de Machine Learning